# TASK MAKER #

In [1]:
import json
import os
import matplotlib.pyplot as plt
import asyncio
import random
from datetime import datetime
from ipywidgets import VBox, HBox, Button, Textarea, FloatText, Output, Dropdown, Label, Accordion, Tab
from IPython.display import display, clear_output, HTML

# --- Files ---
TASKS_FILE = "tasks.json"
COMPLETED_FILE = "completed.json"

# --- Compliment Bank ---
COMPLIMENTS = [
    "Fantastic work!", "You're on a roll!", "Stellar effort!", "That's how it's done!", "Incredible!",
    "Amazing job!", "You're a superstar!", "Keep up the great work!", "Nothing can stop you!", "Outstanding!",
    "You're making great progress!", "Brilliant!", "Excellent!", "Wonderful!", "You've got this!",
    "Impressive!", "Magnificent!", "You're a genius!", "Superb!", "Phenomenal!", "Exceptional!",
    "You're unstoppable!", "Great job!", "Well done!", "Keep it up!", "Proud of you!", "Nailed it!",
    "You rock!", "Mission accomplished!", "Task complete!", "Another one bites the dust!", "Done and done!",
    "That's a win!", "Major accomplishment!", "You're crushing it!", "Look at you go!", "You're on fire!",
    "Way to go!", "That's the way!", "You're a legend!", "Top-notch work!", "First-class effort!",
    "Five-star performance!", "A+ work!", "You're a true professional!", "Masterful!", "Skillful!",
    "Expertly done!", "A job well done!", "You make it look easy!", "You're a natural!", "Simply the best!",
    "Better than ever!", "You've outdone yourself!", "A true masterpiece!", "A work of art!", "Perfection!",
    "Couldn't have done it better myself!", "You're an inspiration!", "You're a role model!", "Keep shining!",
    "You're glowing!", "You're radiant!", "You're a beacon of light!", "You're a shining star!", "Reach for the stars!",
    "The sky's the limit!", "To infinity and beyond!", "You're going places!", "The world is your oyster!",
    "You're destined for greatness!", "The future is bright!", "You've got a bright future ahead of you!",
    "This is just the beginning!", "The best is yet to come!", "Onwards and upwards!", "Keep moving forward!",
    "Never give up!", "You're a fighter!", "You're a warrior!", "You're a champion!", "You're a winner!",
    "You're a conqueror!", "You're a victor!", "You're triumphant!", "You're victorious!", "You're a hero!",
    "You're my hero!", "You're an idol!", "You're a rockstar!", "You're a legend in the making!", "History has its eyes on you!",
    "You're making a difference!", "You're changing the world!", "You're making the world a better place!",
    "You're a force for good!", "You're a true inspiration!", "I'm in awe of you!", "I'm speechless!",
    "Words can't describe how proud I am!", "You leave me breathless!", "You take my breath away!", "You're one of a kind!",
    "There's no one else like you!", "You're unique!", "You're special!", "You're a treasure!", "You're a gift!",
    "You're a blessing!", "You're a miracle!", "You're a dream come true!", "You're my everything!", "I'm so lucky to know you!",
    "The world is lucky to have you!", "You're a gift to the world!", "Thank you for being you!", "Never change!",
    "Stay true to yourself!", "Be you, bravely!", "You are enough!", "You are worthy!", "You are loved!",
    "You are valued!", "You are appreciated!", "You are respected!", "You are admired!", "You are adored!",
    "You are cherished!", "You are treasured!", "You are prized!", "You are esteemed!", "You are revered!",
    "You are honored!", "You are celebrated!", "You are glorified!", "You are exalted!", "You are magnificent!",
    "You are divine!", "You are sublime!", "You are supreme!", "You are unparalleled!", "You are unmatched!",
    "You are unrivaled!", "You are second to none!", "You are in a class of your own!", "You are in a league of your own!",
    "You are ahead of the curve!", "You are a trendsetter!", "You are a trailblazer!", "You are a pioneer!",
    "You are a visionary!", "You are a revolutionary!", "You are a game-changer!", "You are a world-changer!",
    "You are a history-maker!", "You are a legend!", "Absolutely killing it!", "That was smooth!", "Unbelievable!",
    "Task obliterated!", "Consider it handled.", "Another goal scored!", "Progress personified!", "Efficiency at its finest.",
    "That's some high-quality work.", "You've set a new standard.", "The bar has been raised.", "Pure dedication on display.",
    "The focus is real.", "Flawless execution.", "A perfect 10!", "You didn't just do it, you owned it.",
    "Making it happen!", "That's the spirit!", "Energy and effort paying off.", "Peak performance!",
    "You're in the zone!", "That's momentum!", "Building a legacy, one task at a time.", "The definition of success.",
    "This is what progress looks like.", "Your hard work is so clear.", "A true display of skill.", "Respect.",
    "Bravo!", "Encore!", "Chef's kiss!", "Simply perfect.", "Another one checked off the list!",
    "You're a machine!", "Productivity level: Expert.", "Just keeps getting better.", "A sight to behold."
]

ENCOURAGEMENTS = [
    "Tackle this first to build amazing momentum!",
    "Your future self will thank you for getting this done.",
    "Just imagine the satisfaction of checking this one off!",
    "This is your chance to make a huge impact.",
    "Getting this done will bring you so much clarity.",
    "You've got the skills to knock this out of the park.",
    "Focus on this and the rest will feel easier.",
    "Completing this will be a massive win for the day!",
    "This is the key to unlocking your progress.",
    "Let's get this done and make today a success story.",
    "Give this your focus and feel the stress melt away.",
    "You're more than capable of handling this challenge.",
    "Start with this one and you'll be unstoppable.",
    "This is where you can really shine.",
    "Think of the peace of mind you'll have after this.",
    "Let's turn this challenge into an achievement.",
    "This task is your stepping stone to a great day.",
    "Crush this goal and set the tone for success.",
    "The sooner you start, the sooner you'll feel amazing.",
    "You are ready for this. Let's do it!"
]


# --- Load/Save ---
def load_tasks():
    if not os.path.exists(TASKS_FILE): return {}
    try:
        with open(TASKS_FILE, "r") as f:
            data = json.load(f)
            migrated = False
            for name, value in data.items():
                if not isinstance(value, dict):
                    data[name] = {"urgency": value, "info": "", "size": 2.5}; migrated = True
                if 'priority' in value:
                    value['urgency'] = value.pop('priority'); migrated = True
                if 'weight' in value:
                    value['size'] = value.pop('weight'); migrated = True
                if 'size' not in value:
                    value['size'] = 2.5; migrated = True
            if migrated: save_tasks(data)
            return data
    except (json.JSONDecodeError, AttributeError): return {}

def save_tasks(tasks):
    with open(TASKS_FILE, "w") as f: json.dump(tasks, f, indent=2)

def load_completed():
    if not os.path.exists(COMPLETED_FILE): return []
    try:
        with open(COMPLETED_FILE, "r") as f:
            completed_data = json.load(f)
            migrated = False
            for entry in completed_data:
                if 'priority' in entry:
                    entry['urgency'] = entry.pop('priority'); migrated = True
                if 'weight' in entry:
                    entry['size'] = entry.pop('weight'); migrated = True
            if migrated:
                with open(COMPLETED_FILE, "w") as fw: json.dump(completed_data, fw, indent=2)
            return completed_data
    except json.JSONDecodeError: return []

def save_completed(task_name, task_data):
    completed_list = load_completed()
    new_entry = {
        "name": task_name, "urgency": task_data.get('urgency', 'N/A'),
        "info": task_data.get('info', ''), "size": task_data.get('size', 2.5),
        "completed_at": datetime.now().isoformat()
    }
    completed_list.append(new_entry)
    with open(COMPLETED_FILE, "w") as f: json.dump(completed_list, f, indent=2)

# --- Widgets Setup ---
tasks = load_tasks()
out = Output()
graph_out = Output()

# Primary Widgets
multi_task_input = Textarea(
    placeholder="Task, Urgency (0-10), Size (1-5), [Optional Info]\ne.g., Write proposal, 9, 4.5, For the new project",
    layout={'width': '500px', 'height': '120px'}
)
add_button = Button(description="Add Tasks", button_style="success")
graph_button = Button(description="Show Graph", button_style="warning")
hide_graph_button = Button(description="Hide Graph")
show_all_button = Button(description="Show All Tasks", button_style="info") # NEW
completed_button = Button(description="Show Completed", button_style="primary")
stats_button = Button(description="Show Stats", button_style="info")

# Widgets for editing tasks
edit_task_dropdown = Dropdown(description="Task:")
new_urgency_input = FloatText(description="New Urgency:", layout={'width': '200px'})
update_urgency_button = Button(description="Update", button_style="primary")
new_size_input = FloatText(description="New Size (1-5):", layout={'width': '200px'})
update_size_button = Button(description="Update", button_style="primary")
new_info_input = Textarea(placeholder="Enter new additional info here.", layout={'width': '400px', 'height': '80px'})
update_info_button = Button(description="Update", button_style="primary")
delete_button = Button(description="Delete Task", button_style="danger")

# Widgets for completing and managing completed tasks
complete_dropdown = Dropdown(description="Task:")
complete_button = Button(description="Complete", button_style="info")
delete_completed_dropdown = Dropdown(description="Completed Task:")
delete_completed_button = Button(description="Delete From History", button_style="danger")


# --- Functions ---
def refresh_completed_dropdown():
    completed_list = sorted(load_completed(), key=lambda x: x['completed_at'], reverse=True)
    comp_options = []
    for entry in completed_list:
        dt = datetime.fromisoformat(entry['completed_at'])
        date_str = dt.strftime("%Y-%m-%d %H:%M")
        comp_options.append((f"{entry['name']} ({date_str})", entry['completed_at']))
    if not comp_options:
        comp_options = [("No completed tasks", None)]
    delete_completed_dropdown.options = comp_options
    
def refresh_all_dropdowns():
    sorted_items = sorted(tasks.items(), key=lambda x: x[1]['urgency'], reverse=True)
    dropdown_options = [(f"{name} (U: {data['urgency']}, S: {data.get('size', 'N/A')})", name) for name, data in sorted_items]
    if not dropdown_options: dropdown_options = [("No tasks available", None)]
    edit_task_dropdown.options = dropdown_options
    complete_dropdown.options = dropdown_options
    refresh_completed_dropdown()

def add_task(b):
    out.clear_output(wait=True)
    with out:
        lines = multi_task_input.value.strip().split('\n')
        added, updated, errors = 0, 0, []
        if not multi_task_input.value.strip():
            print("⚠️ Input area is empty.")
            return
        for i, line in enumerate(lines):
            if not line.strip(): continue
            parts = [p.strip() for p in line.split(',', 3)]
            if len(parts) < 3: errors.append(f"L{i+1}: Format must be 'Task, Urgency, Size'. -> '{line}'"); continue
            name, urgency_str, size_str = parts[0], parts[1], parts[2]
            info = parts[3] if len(parts) > 3 else ""
            if not name: errors.append(f"L{i+1}: Task name empty. -> '{line}'"); continue
            try: urgency = float(urgency_str)
            except ValueError: errors.append(f"L{i+1}: Invalid urgency number. -> '{line}'"); continue
            try: 
                size = float(size_str)
            except ValueError: errors.append(f"L{i+1}: Invalid size number. -> '{line}'"); continue
            if not (0 <= urgency <= 10): errors.append(f"L{i+1}: Urgency must be 0-10. -> '{line}'"); continue
            if not (1 <= size <= 5): errors.append(f"L{i+1}: Size must be 1-5. -> '{line}'"); continue
            if name in tasks: updated += 1
            else: added += 1
            tasks[name] = {"urgency": round(urgency, 2), "info": info, "size": round(size, 2)}
        if added > 0 or updated > 0:
            save_tasks(tasks)
            refresh_all_dropdowns()
            show_graph(None) 
        multi_task_input.value = ""
        
        if added > 0: print(f"✅ Added {added} new task(s).")
        if updated > 0: print(f"🔄 Updated {updated} task(s).")
        if not errors and added == 0 and updated == 0: print("No changes made.")
        if errors: print("\n--- Errors ---\n" + "\n".join(f"⚠️ {e}" for e in errors))

def update_task_urgency(b):
    out.clear_output(wait=True)
    with out:
        task_name = edit_task_dropdown.value
        if not task_name:
            print("⚠️ No task selected.")
            return
        new_urgency = new_urgency_input.value
        if not (0 <= new_urgency <= 10):
            print("⚠️ Urgency must be 0-10.")
            return
        tasks[task_name]['urgency'] = round(new_urgency, 2)
        save_tasks(tasks); refresh_all_dropdowns(); new_urgency_input.value = 0
        print(f"✅ Urgency for '{task_name}' updated to {new_urgency}.")

def update_task_size(b):
    out.clear_output(wait=True)
    with out:
        task_name = edit_task_dropdown.value
        if not task_name:
            print("⚠️ No task selected.")
            return
        new_size = new_size_input.value
        if not (1 <= new_size <= 5):
            print("⚠️ Size must be a number from 1-5.")
            return
        tasks[task_name]['size'] = round(new_size, 2)
        save_tasks(tasks); refresh_all_dropdowns(); new_size_input.value = 0
        print(f"✅ Size for '{task_name}' updated to {round(new_size, 2)}.")

def populate_info_editor(change):
    task_name = change.new
    if task_name in tasks:
        task_data = tasks[task_name]
        new_info_input.value = task_data.get('info', '')
        new_urgency_input.value = task_data.get('urgency', 0)
        new_size_input.value = task_data.get('size', 2.5)

def update_task_info(b):
    out.clear_output(wait=True)
    with out:
        task_name = edit_task_dropdown.value
        if not task_name:
            print("⚠️ No task selected.")
            return
        tasks[task_name]['info'] = new_info_input.value.strip()
        save_tasks(tasks); new_info_input.value = ""
        print(f"✅ Info for '{task_name}' has been updated.")

def delete_task(b):
    out.clear_output(wait=True)
    with out:
        task_name = edit_task_dropdown.value
        if not task_name:
            print("⚠️ No task selected to delete.")
            return
        
        tasks.pop(task_name, None)
        save_tasks(tasks); refresh_all_dropdowns(); show_graph(None)
        print(f"🗑️ Task '{task_name}' has been deleted.")

async def run_completion_animation(task_name):
    frames = ["🎆", "🎇", "✨", "🎉", "🎊", "🌟", "🏆", "🥇"]
    display_line = ""
    with out:
        for i in range(15):
            clear_output(wait=True)
            display_line += random.choice(frames)
            print(f"Completed: {task_name}\n\n{display_line}")
            await asyncio.sleep(0.12)
        
        clear_output(wait=True)
        compliment = random.choice(COMPLIMENTS)
        
        html_reward = f"""
        <style>
            @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;600&display=swap');
            .reward-box {{
                font-family: 'Poppins', sans-serif;
                background: linear-gradient(145deg, #f8e4a2, #e9c46a);
                border: 2px solid #d4af37;
                border-radius: 12px;
                padding: 25px;
                text-align: center;
                box-shadow: 0 4px 15px rgba(0, 0, 0, 0.2);
                max-width: 450px;
                margin: 20px auto;
                animation: fadeIn 0.5s ease-in-out;
            }}
            @keyframes fadeIn {{
                from {{ opacity: 0; transform: scale(0.9); }}
                to {{ opacity: 1; transform: scale(1); }}
            }}
            .reward-title {{
                font-size: 24px;
                font-weight: 600;
                color: #2a2a2a;
                margin-bottom: 10px;
            }}
            .reward-subtitle {{
                font-size: 16px;
                font-weight: 400;
                color: #3d3d3d;
                margin-bottom: 15px;
            }}
            .reward-text {{
                font-size: 18px;
                font-style: italic;
                color: #333;
            }}
        </style>
        <div class="reward-box">
            <div class="reward-title">🏆 Well Done! 🏆</div>
            <div class="reward-subtitle">You completed: <strong>{task_name}</strong></div>
            <div class="reward-text">"{compliment}"</div>
        </div>
        """
        display(HTML(html_reward))


def complete_task(b):
    task_name = complete_dropdown.value
    if not task_name:
        out.clear_output(wait=True)
        with out: print("⚠️ No task to complete.")
        return
    task_data = tasks.pop(task_name, None)
    if task_data:
        save_tasks(tasks); save_completed(task_name, task_data); refresh_all_dropdowns()
        show_graph(None); asyncio.create_task(run_completion_animation(task_name))

def delete_completed_task(b):
    out.clear_output(wait=True)
    with out:
        task_id = delete_completed_dropdown.value
        if not task_id:
            print("⚠️ No completed task selected to delete.")
            return

        completed_list = load_completed()
        task_to_delete = None
        for task in completed_list:
            if task['completed_at'] == task_id:
                task_to_delete = task
                break
        
        if task_to_delete:
            completed_list.remove(task_to_delete)
            with open(COMPLETED_FILE, "w") as f:
                json.dump(completed_list, f, indent=2)
            
            refresh_completed_dropdown()
            print(f"🗑️ Removed '{task_to_delete['name']}' from your history.")
        else:
            print("Error: Could not find the selected task to delete.")


def show_graph(b):
    out.clear_output(wait=True)
    if not tasks:
        with graph_out: clear_output()
        with out: print("📊 No active tasks to graph."); return

    with out:
        sorted_tasks_for_msg = sorted(tasks.items(), key=lambda x: x[1]['urgency'], reverse=True)
        
        top_tasks_to_exclude = set()
        
        top_task_name = sorted_tasks_for_msg[0][0]
        top_tasks_to_exclude.add(top_task_name)
        encouragement = random.choice(ENCOURAGEMENTS)
        
        if len(sorted_tasks_for_msg) > 1:
            second_task_name = sorted_tasks_for_msg[1][0]
            top_tasks_to_exclude.add(second_task_name)
            task_message = f"Start with <strong>{top_task_name}</strong>, then move on to <strong>{second_task_name}</strong>."
        else:
            task_message = f"Your main focus should be: <strong>{top_task_name}</strong>."
            
        low_size_tasks = [
            name for name, data in tasks.items() 
            if data.get('size', 2.5) <= 2 and name not in top_tasks_to_exclude
        ]
        
        bonus_task_message = ""
        if low_size_tasks:
            bonus_task = random.choice(low_size_tasks)
            bonus_task_message = f"""
            <div class="bonus-task">
                While you're at it, it would be great if you got <strong>{bonus_task}</strong> done!
            </div>
            """

        focus_html = f"""
        <style>
            @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;600&display=swap');
            .focus-box {{
                font-family: 'Poppins', sans-serif;
                background: linear-gradient(145deg, #e0f7fa, #b2ebf2);
                border: 2px solid #4dd0e1;
                border-radius: 12px;
                padding: 25px;
                text-align: center;
                box-shadow: 0 4px 15px rgba(0, 0, 0, 0.1);
                max-width: 500px;
                margin: 0 auto 20px auto;
                animation: fadeIn 0.5s ease-in-out;
            }}
            .focus-title {{
                font-size: 22px;
                font-weight: 600;
                color: #00796b;
                margin-bottom: 10px;
            }}
            .focus-tasks {{
                font-size: 16px;
                color: #004d40;
                margin-bottom: 15px;
            }}
            .focus-encouragement {{
                font-size: 18px;
                font-style: italic;
                color: #00695c;
                margin-bottom: 15px;
            }}
            .bonus-task {{
                font-size: 14px;
                color: #004d40;
                border-top: 1px dashed #00796b;
                padding-top: 10px;
                margin-top: 15px;
            }}
        </style>
        <div class="focus-box">
            <div class="focus-title">🎯 Your Top Priorities! 🎯</div>
            <div class="focus-tasks">{task_message}</div>
            <div class="focus-encouragement">"{encouragement}"</div>
            {bonus_task_message}
        </div>
        """
        display(HTML(focus_html))
        
    with graph_out:
        clear_output(wait=True)
        sorted_tasks = sorted(tasks.items(), key=lambda x: x[1]['urgency'], reverse=True)
        names = [i[0] for i in sorted_tasks]
        scores = [i[1]['urgency'] for i in sorted_tasks]
        sizes = [i[1].get('size', 2.5) for i in sorted_tasks]
        
        min_width, max_width = 0.4, 1.0
        bar_widths = [min_width + (s - 1) * (max_width - min_width) / 4 for s in sizes]

        plt.style.use("seaborn-v0_8-talk"); fig, ax = plt.subplots(figsize=(10, 6))
        colors = []; num = len(names); red = '#d90429'
        if num >= 1: colors.append(red);
        if num >= 2: colors.append(red)
        rem = num - 2
        if rem > 0:
            cmap = plt.colormaps.get('viridis_r')
            for i in range(rem): colors.append(cmap(((i / (rem - 1)) if rem > 1 else 0.0) * 0.8))
        
        bars = ax.bar(names, scores, color=colors, width=bar_widths)
        
        ranks = [f"{i+1}{('st' if (i+1)%10==1 and (i+1)%100!=11 else 'nd' if (i+1)%10==2 and (i+1)%100!=12 else 'rd' if (i+1)%10==3 and (i+1)%100!=13 else 'th')}" for i in range(len(names))]
        ax.bar_label(bars, labels=ranks, padding=3, fontsize=11, fontweight='bold')
        ax.set_ylim(0, 11.5); ax.set_ylabel("Urgency Score"); ax.set_title("Active Task Urgency & Size")
        fig.tight_layout(); plt.xticks(rotation=40, ha="right"); plt.show()

def show_all_tasks(b):
    # NEW: Displays a simple text list of all active tasks.
    out.clear_output(wait=True)
    with out:
        if not tasks:
            print("No active tasks to show.")
            return
        
        print("--- All Active Tasks ---")
        sorted_tasks_list = sorted(tasks.items(), key=lambda x: x[1]['urgency'], reverse=True)
        for name, data in sorted_tasks_list:
            urgency = data.get('urgency', 'N/A')
            size = data.get('size', 'N/A')
            info = data.get('info', '')
            info_str = f" | Info: {info}" if info else ""
            print(f"- {name} (Urgency: {urgency}, Size: {size}){info_str}")

def show_completed(b):
    out.clear_output(wait=True)
    with out:
        comp = load_completed()
        if not comp:
            print("No tasks completed yet."); return

        tasks_by_date = {}
        for entry in comp:
            dt = datetime.fromisoformat(entry['completed_at'])
            date_str = dt.strftime("%Y-%m-%d")
            if date_str not in tasks_by_date: tasks_by_date[date_str] = []
            tasks_by_date[date_str].append(entry)

        print("--- Completed Tasks ---")
        sorted_dates = sorted(tasks_by_date.keys(), reverse=True)
        for date_str in sorted_dates:
            print(f"\n--- {date_str} ---")
            day_tasks = sorted(tasks_by_date[date_str], key=lambda x: x['completed_at'], reverse=True)
            for entry in day_tasks:
                dt = datetime.fromisoformat(entry['completed_at'])
                time_str = dt.strftime("%H:%M")
                size = f" (Size: {entry.get('size', 'N/A')})"
                info = f" | Info: {entry['info']}" if entry.get('info') else ""
                print(f"  {time_str} - ✔ {entry['name']}{size}{info}")

def show_stats(b):
    out.clear_output(wait=True)
    with out:
        completed_list = load_completed()
        achieved_size = sum(task.get('size', 2.5) for task in completed_list)
        active_size = sum(data.get('size', 2.5) for data in tasks.values())
        total_possible_size = achieved_size + active_size
        if total_possible_size == 0:
            print("📊 No tasks yet. Add some to see your stats based on task size!"); return
        completion_perc = (achieved_size / total_possible_size) * 100
        print("--- Your Progress Stats (Based on Task Size) ---")
        print(f"✅ Completed Effort (Size): {achieved_size}")
        print(f"⏳ Remaining Effort (Size): {active_size}")
        print("--------------------------------------")
        print(f"📈 Completion Rate: {completion_perc:.2f}%")
        bar_length = 30
        filled_length = int(bar_length * completion_perc / 100)
        bar = '█' * filled_length + '-' * (bar_length - filled_length)
        print(f"\nOverall Progress:\n[{bar}]")

def hide_graph(b):
    out.clear_output()
    graph_out.clear_output()

# --- Event Binding ---
add_button.on_click(add_task)
update_urgency_button.on_click(update_task_urgency)
update_size_button.on_click(update_task_size)
delete_button.on_click(delete_task)
edit_task_dropdown.observe(populate_info_editor, names='value')
update_info_button.on_click(update_task_info)
complete_button.on_click(complete_task)
delete_completed_button.on_click(delete_completed_task)
graph_button.on_click(show_graph)
hide_graph_button.on_click(hide_graph)
show_all_button.on_click(show_all_tasks) # NEW
completed_button.on_click(show_completed)
stats_button.on_click(show_stats)

# --- GUI Layout ---
refresh_all_dropdowns()

task_input_box = HBox([multi_task_input, add_button])
primary_buttons_box = HBox([graph_button, hide_graph_button, completed_button, show_all_button, stats_button]) # UPDATED

# Tab widget for editing tasks
edit_urgency_box = HBox([new_urgency_input, update_urgency_button])
edit_size_box = HBox([new_size_input, update_size_button])
edit_info_box = VBox([new_info_input, update_info_button])
delete_task_box = VBox([Label("This action is permanent."), delete_button])
edit_tabs = Tab(children=[edit_urgency_box, edit_size_box, edit_info_box, delete_task_box])
edit_tabs.set_title(0, 'Edit Urgency'); edit_tabs.set_title(1, 'Edit Size')
edit_tabs.set_title(2, 'Edit Info'); edit_tabs.set_title(3, 'Delete Task')
edit_task_box = VBox([Label("Select a task to manage:"), edit_task_dropdown, edit_tabs])

# Widgets for completing and managing history
complete_box = HBox([complete_dropdown, complete_button])
delete_completed_box = HBox([delete_completed_dropdown, delete_completed_button])

# Accordion for managing tasks (excluding completing)
management_accordion = Accordion(children=[edit_task_box, delete_completed_box])
management_accordion.set_title(0, 'Manage Active Tasks')
management_accordion.set_title(1, 'Manage History')

# Final display assembly
display(VBox([
    Label("Add/Update Tasks (Task, Urgency, Size, [Optional Info]):"),
    task_input_box,
    Label(""), # Spacer
    Label("Complete a Task:"),
    complete_box,
    primary_buttons_box,
    out,
    graph_out,
    Label(""), # Spacer
    management_accordion
]))



# SCHEDULE MAKER

In [2]:
import json
import os
from datetime import datetime, timedelta, time
from ipywidgets import VBox, HBox, Button, Textarea, DatePicker, Output, Label, IntText, Dropdown, Tab
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --- Constants and Configuration ---
TASKS_FILE = "tasks.json"
SCHEDULE_FILE = "schedule_data.json"

# --- Helper Functions ---
def load_tasks():
    """Loads tasks from the JSON file, handling potential errors and migration."""
    if not os.path.exists(TASKS_FILE): return {}
    try:
        with open(TASKS_FILE, "r") as f:
            data = json.load(f)
            migrated = False
            for name, value in data.items():
                if not isinstance(value, dict): data[name] = {"urgency": value, "info": "", "size": 2.5}; migrated = True
                if 'priority' in value: value['urgency'] = value.pop('priority'); migrated = True
                if 'weight' in value: value['size'] = value.pop('weight'); migrated = True
            if migrated:
                 with open(TASKS_FILE, "w") as fw: json.dump(data, fw, indent=2)
            return data
    except (json.JSONDecodeError, AttributeError): return {}

def load_schedule_data():
    """Loads busy slots, repeating slots, and manual schedule from a JSON file."""
    if not os.path.exists(SCHEDULE_FILE):
        return {}, {}, []
    try:
        with open(SCHEDULE_FILE, "r") as f:
            data = json.load(f)
            
            busy_slots = {}
            raw_busy_data = data.get("busy_slots", {})
            for date_str, slots in raw_busy_data.items():
                date_obj = datetime.fromisoformat(date_str).date()
                busy_slots[date_obj] = []
                for slot in slots:
                    if len(slot) == 2:
                        s, e = slot
                        busy_slots[date_obj].append((datetime.strptime(s, "%H:%M").time(), datetime.strptime(e, "%H:%M").time(), "Busy"))
                    else:
                        s, e, label = slot
                        busy_slots[date_obj].append((datetime.strptime(s, "%H:%M").time(), datetime.strptime(e, "%H:%M").time(), label))

            manual_schedule = {}
            raw_manual_data = data.get("manual_schedule", {})
            for date_str, slots in raw_manual_data.items():
                date_obj = datetime.fromisoformat(date_str).date()
                manual_schedule[date_obj] = []
                for slot in slots:
                    if len(slot) == 4:
                        s, e, n, u = slot
                        manual_schedule[date_obj].append((datetime.fromisoformat(s), datetime.fromisoformat(e), n, u, 'manual'))
                    else:
                        s, e, n, u, source = slot
                        manual_schedule[date_obj].append((datetime.fromisoformat(s), datetime.fromisoformat(e), n, u, source))

            repeating_slots_raw = data.get("repeating_busy_slots", [])
            repeating_slots = [(datetime.strptime(s, "%H:%M").time(), datetime.strptime(e, "%H:%M").time(), label) for s, e, label in repeating_slots_raw]
            
            return busy_slots, manual_schedule, repeating_slots
    except (json.JSONDecodeError, KeyError):
        return {}, {}, []

def save_schedule_data():
    """Saves all schedule data to a JSON file."""
    busy_slots_str = {d.isoformat(): [(s.strftime("%H:%M"), e.strftime("%H:%M"), label) for s, e, label in slots] for d, slots in busy_slots.items()}
    manual_schedule_str = {d.isoformat(): [(s.isoformat(), e.isoformat(), n, u, source) for s, e, n, u, source in slots] for d, slots in manual_schedule.items()}
    repeating_slots_str = [(s.strftime("%H:%M"), e.strftime("%H:%M"), label) for s, e, label in repeating_busy_slots]
    
    with open(SCHEDULE_FILE, "w") as f:
        json.dump({
            "busy_slots": busy_slots_str, 
            "manual_schedule": manual_schedule_str,
            "repeating_busy_slots": repeating_slots_str
        }, f, indent=2)

def size_to_duration_minutes(size):
    """Converts a task's size (1-5) into an estimated duration in minutes."""
    return int(size * 45 + 15)

def merge_busy_slots(slot_list):
    """NEW: Merges overlapping busy intervals into single blocks."""
    if not slot_list:
        return []
    
    # Sort slots by start time
    slot_list.sort(key=lambda x: x[0])
    
    merged = [list(slot_list[0])] # Start with the first slot
    
    for current_start, current_end, current_label in slot_list[1:]:
        last_start, last_end, last_label = merged[-1]
        
        # If they overlap, merge them
        if current_start < last_end:
            # Update the end time of the last merged slot
            merged[-1][1] = max(last_end, current_end)
            # The label of the earliest starting slot is kept
        else:
            merged.append([current_start, current_end, current_label])
            
    return [tuple(slot) for slot in merged]

# --- Widget Setup ---
busy_slot_date_picker = DatePicker(description="For Date:", value=datetime.now().date())
busy_start_input = Textarea(placeholder="Start (e.g., 13:00)", layout={'width': '120px', 'height': '30px'})
busy_end_input = Textarea(placeholder="End (e.g., 14:30)", layout={'width': '120px', 'height': '30px'})
busy_label_input = Textarea(placeholder="Label (e.g., Lunch)", layout={'width': '150px', 'height': '30px'})
add_busy_slot_button = Button(description="Add Specific Slot", button_style="primary")
generate_schedule_button = Button(description="Generate 7-Day Schedule", button_style="success", layout={'width': '220px'})
clear_schedule_button = Button(description="Clear All Scheduled Data", button_style="danger")
busy_slots_out = Output()
schedule_out = Output()
manual_task_dropdown = Dropdown(description="Task:")
manual_date_picker = DatePicker(description="On Date:", value=datetime.now().date())
manual_start_time_input = Textarea(placeholder="Start Time (e.g., 10:00)", layout={'width': '180px', 'height': '30px'})
manual_schedule_button = Button(description="Schedule Manually", button_style="info")
repeating_start_input = Textarea(placeholder="Start (e.g., 12:00)", layout={'width': '120px', 'height': '30px'})
repeating_end_input = Textarea(placeholder="End (e.g., 13:00)", layout={'width': '120px', 'height': '30px'})
repeating_label_input = Textarea(placeholder="Label (e.g., Lunch)", layout={'width': '150px', 'height': '30px'})
add_repeating_slot_button = Button(description="Add Daily Slot", button_style="primary")
remove_slot_dropdown = Dropdown(description="Select Slot to Remove:")
remove_slot_button = Button(description="Remove Selected Slot", button_style="danger")


# --- Global State ---
session_tasks = load_tasks() 
busy_slots, manual_schedule, repeating_busy_slots = load_schedule_data()

# --- Functions ---
def refresh_removal_dropdown():
    """Populates the dropdown for removing fixed slots."""
    options = []
    for i, (start, end, label) in enumerate(repeating_busy_slots):
        text = f"DAILY: {label} ({start.strftime('%H:%M')}-{end.strftime('%H:%M')})"
        options.append((text, ('repeating', i)))
    for date, slots in sorted(busy_slots.items()):
        for i, (start, end, label) in enumerate(slots):
            text = f"{date.strftime('%a, %b %d')}: {label} ({start.strftime('%H:%M')}-{end.strftime('%H:%M')})"
            options.append((text, ('specific', date.isoformat(), i)))
    for date, slots in sorted(manual_schedule.items()):
        for i, (start, end, name, *rest) in enumerate(slots):
            text = f"{date.strftime('%a, %b %d')}: MANUAL - {name} ({start.strftime('%H:%M')}-{end.strftime('%H:%M')})"
            options.append((text, ('manual', date.isoformat(), i)))
    
    if not options:
        options = [("No fixed slots to remove", None)]
    remove_slot_dropdown.options = options


def refresh_manual_task_dropdown():
    """Updates the dropdown with tasks not yet manually scheduled."""
    manually_scheduled_names = {name for date in manual_schedule for _, _, name, _, _ in manual_schedule[date]}
    available_tasks = {name: data for name, data in load_tasks().items() if name not in manually_scheduled_names}
    
    task_options = [(name, name) for name in sorted(available_tasks.keys())]
    if not task_options:
        task_options = [("No tasks available to schedule", None)]
    manual_task_dropdown.options = task_options

def display_fixed_times():
    """Refreshes the display of all fixed time slots."""
    with busy_slots_out:
        clear_output(wait=True)
        has_any_fixed_time = busy_slots or manual_schedule or repeating_busy_slots
        if not has_any_fixed_time:
            print("No busy or manually scheduled times added yet.")
            return
            
        print("--- Your Fixed Times ---")
        if repeating_busy_slots:
            print("\nRepeating Daily Slots:")
            for start, end, label in sorted(repeating_busy_slots):
                print(f"  - {label} from {start.strftime('%H:%M')} to {end.strftime('%H:%M')}")

        all_dates = set(busy_slots.keys()) | set(manual_schedule.keys())
        for date in sorted(list(all_dates)):
            print(f"\nOn {date.strftime('%A')}:")
            if date in busy_slots:
                for start, end, label in sorted(busy_slots[date]):
                    print(f"  - {label} from {start.strftime('%H:%M')} to {end.strftime('%H:%M')}")
            if date in manual_schedule:
                for start, end, name, *rest in sorted(manual_schedule[date]):
                    print(f"  - MANUAL: {start.strftime('%H:%M')}-{end.strftime('%H:%M')} - {name}")
    refresh_removal_dropdown()

def add_busy_slot(b):
    """Handles adding a new specific busy time slot."""
    start_str, end_str = busy_start_input.value.strip(), busy_end_input.value.strip()
    label = busy_label_input.value.strip() or "Busy"
    slot_date = busy_slot_date_picker.value
    try:
        start_time, end_time = datetime.strptime(start_str, "%H:%M").time(), datetime.strptime(end_str, "%H:%M").time()
        if start_time >= end_time:
            with schedule_out: clear_output(); print(f"⚠️ Error: Start time must be before end time.")
            return
        if slot_date not in busy_slots: busy_slots[slot_date] = []
        busy_slots[slot_date].append((start_time, end_time, label))
        save_schedule_data()
        busy_start_input.value, busy_end_input.value, busy_label_input.value = "", "", ""
        display_fixed_times()
        with schedule_out: clear_output()
    except ValueError:
        with schedule_out: clear_output(); print("⚠️ Error: Invalid time format. Use HH:MM.")

def add_repeating_busy_slot(b):
    """Handles adding a new repeating daily busy slot."""
    start_str, end_str = repeating_start_input.value.strip(), repeating_end_input.value.strip()
    label = repeating_label_input.value.strip() or "Daily Block"
    try:
        start_time, end_time = datetime.strptime(start_str, "%H:%M").time(), datetime.strptime(end_str, "%H:%M").time()
        if start_time >= end_time:
            with schedule_out: clear_output(); print("⚠️ Error: Start time must be before end time.")
            return
        repeating_busy_slots.append((start_time, end_time, label))
        save_schedule_data()
        repeating_start_input.value, repeating_end_input.value, repeating_label_input.value = "", "", ""
        display_fixed_times()
        with schedule_out: clear_output()
    except ValueError:
        with schedule_out: clear_output(); print("⚠️ Error: Invalid time format. Use HH:MM.")

def manual_schedule_task(b):
    """Manually schedules a single task into a specific time slot and saves it."""
    with schedule_out:
        clear_output(wait=True)
        task_name = manual_task_dropdown.value
        if not task_name: print("⚠️ Please select a task."); return
        
        task_data = load_tasks().get(task_name)
        if not task_data: print(f"⚠️ Task '{task_name}' no longer exists."); return
            
        date = manual_date_picker.value
        start_time_str = manual_start_time_input.value.strip()
        try:
            start_time = datetime.strptime(start_time_str, "%H:%M").time()
        except ValueError:
            print("⚠️ Invalid start time format. Use HH:MM."); return
            
        duration = timedelta(minutes=size_to_duration_minutes(task_data.get('size', 2.5)))
        task_start_dt = datetime.combine(date, start_time)
        task_end_dt = task_start_dt + duration
        
        all_day_fixed_slots = busy_slots.get(date, []) + repeating_busy_slots
        for busy_start_t, busy_end_t, _ in all_day_fixed_slots:
            busy_start_dt = datetime.combine(date, busy_start_t)
            busy_end_dt = datetime.combine(date, busy_end_t)
            if max(task_start_dt, busy_start_dt) < min(task_end_dt, busy_end_dt):
                print(f"⚠️ Conflict! Overlaps with a busy slot on {date.strftime('%A')}."); return

        if date not in manual_schedule: manual_schedule[date] = []
        manual_schedule[date].append((task_start_dt, task_end_dt, task_name, task_data['urgency'], 'manual'))
        save_schedule_data()
        
        print(f"✅ Manually scheduled '{task_name}' for {date.strftime('%A')} at {start_time_str}.")
        manual_start_time_input.value = ""
        refresh_manual_task_dropdown()
        display_fixed_times()

def remove_fixed_slot(b):
    """Removes a selected fixed slot (busy, repeating, or manual)."""
    with schedule_out:
        clear_output(wait=True)
        slot_id = remove_slot_dropdown.value
        if not slot_id:
            print("⚠️ No slot selected to remove.")
            return

        slot_type = slot_id[0]
        if slot_type == 'repeating':
            index = slot_id[1]
            removed_item = repeating_busy_slots.pop(index)
            print(f"🗑️ Removed repeating slot: {removed_item[2]}")
        elif slot_type == 'specific':
            date = datetime.fromisoformat(slot_id[1]).date()
            index = slot_id[2]
            removed_item = busy_slots[date].pop(index)
            if not busy_slots[date]: del busy_slots[date]
            print(f"🗑️ Removed specific slot: {removed_item[2]} on {date.strftime('%A')}")
        elif slot_type == 'manual':
            date = datetime.fromisoformat(slot_id[1]).date()
            index = slot_id[2]
            removed_item = manual_schedule[date].pop(index)
            if not manual_schedule[date]: del manual_schedule[date]
            print(f"🗑️ Removed manual task: {removed_item[2]} on {date.strftime('%A')}")
        
        save_schedule_data()
        refresh_manual_task_dropdown()
        display_fixed_times()

def clear_schedule_data(b):
    """Clears all busy slots and manual schedules from memory and file."""
    global busy_slots, manual_schedule, repeating_busy_slots
    with schedule_out:
        clear_output(wait=True)
        busy_slots, manual_schedule, repeating_busy_slots = {}, {}, []
        save_schedule_data()
        refresh_manual_task_dropdown()
        display_fixed_times()
        print("🗑️ All busy times and manually scheduled tasks have been cleared.")

def generate_schedule(b):
    """The main logic for creating and displaying the 7-day schedule."""
    with schedule_out:
        clear_output(wait=True)
        all_tasks_from_file = load_tasks()
        manually_scheduled_names = {name for date in manual_schedule for _, _, name, _, _ in manual_schedule[date]}
        auto_tasks_pool_dict = {name: data for name, data in all_tasks_from_file.items() if name not in manually_scheduled_names}
        tasks_pool = sorted(auto_tasks_pool_dict.items(), key=lambda x: x[1]['urgency'], reverse=True)
        
        final_schedule = {}
        start_date = datetime.now().date()
        num_days = 7

        for day_offset in range(num_days):
            current_date = start_date + timedelta(days=day_offset)
            final_schedule[current_date] = []
            final_schedule[current_date].extend(manual_schedule.get(current_date, []))
            
            daily_size_total = sum(load_tasks().get(name, {}).get('size', 2.5) for _, _, name, _, _ in manual_schedule.get(current_date, []))

            if day_offset == 0:
                effective_start_time = max(datetime.combine(current_date, time(8, 0)), datetime.now())
            else:
                effective_start_time = datetime.combine(current_date, time(8, 0))

            work_end = datetime.combine(current_date, time(22, 0))
            
            if effective_start_time < work_end:
                free_slots_today = [(effective_start_time, work_end)]
            else:
                free_slots_today = []

            # UPDATED: Merge busy slots before calculating free time
            all_busy_slots_for_day_raw = \
                [(datetime.combine(current_date, s), datetime.combine(current_date, e), l) for s, e, l in busy_slots.get(current_date, [])] + \
                [(datetime.combine(current_date, s), datetime.combine(current_date, e), l) for s, e, l in repeating_busy_slots]
            
            merged_busy_slots = merge_busy_slots(all_busy_slots_for_day_raw)
            
            all_fixed_slots = merged_busy_slots + [(s, e) for s, e, n, u, source in manual_schedule.get(current_date, [])]

            for fixed_start, fixed_end, *_ in sorted(all_fixed_slots):
                new_free_slots = []
                for free_start, free_end in free_slots_today:
                    if fixed_end <= free_start or fixed_start >= free_end: new_free_slots.append((free_start, free_end)); continue
                    if fixed_start <= free_start and fixed_end >= free_end: continue
                    if fixed_start > free_start: new_free_slots.append((free_start, fixed_start))
                    if fixed_end < free_end: new_free_slots.append((fixed_end, free_end))
                free_slots_today = new_free_slots
            
            for name, data in list(tasks_pool):
                task_size = data.get('size', 2.5)
                if daily_size_total + task_size > 9:
                    continue 
                
                duration = timedelta(minutes=size_to_duration_minutes(task_size))
                
                for i, (free_start, free_end) in enumerate(free_slots_today):
                    if free_end - free_start >= duration:
                        task_start_time = free_start
                        task_end_time = free_start + duration
                        final_schedule[current_date].append((task_start_time, task_end_time, name, data['urgency'], 'auto'))
                        daily_size_total += task_size
                        
                        next_block_start = task_end_time + timedelta(minutes=30)
                        
                        if next_block_start < free_end:
                            free_slots_today[i] = (next_block_start, free_end)
                        else:
                            free_slots_today.pop(i)
                        
                        tasks_pool.remove((name, data))
                        break
        
        plot_schedule(final_schedule, start_date)
        unscheduled_tasks = [name for name, data in tasks_pool]
        if unscheduled_tasks:
            print("\n--- ⚠️ Could Not Schedule (Tasks Remaining) ---")
            for name in unscheduled_tasks: print(f"- {name}")

def plot_schedule(schedule, start_date):
    """Generates a vertical Gantt-style chart with time passing downwards."""
    num_days = 7
    fig, ax = plt.subplots(figsize=(3.5 * num_days, 14))
    
    dates = [start_date + timedelta(days=i) for i in range(num_days)]
    x_labels = [d.strftime('%A') for d in dates]
    ax.set_xticks(range(len(x_labels)))
    ax.set_xticklabels(x_labels, rotation=45, ha='right')
    ax.set_xlim(-0.5, num_days - 0.5)
    
    y_start = datetime.combine(start_date, time(7, 30))
    y_end = datetime.combine(start_date, time(22, 30))
    ax.set_ylim(y_end, y_start)
    ax.yaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.yaxis.set_major_locator(mdates.HourLocator(interval=1))
    ax.set_ylabel("Time of Day")
    
    manual_color = '#3a86ff'
    auto_color = '#ffbe0b'

    for i, date in enumerate(dates):
        # UPDATED: Merge busy slots for plotting
        all_busy_slots_for_day_raw = \
            [(datetime.combine(date, s), datetime.combine(date, e), l) for s, e, l in busy_slots.get(date, [])] + \
            [(datetime.combine(date, s), datetime.combine(date, e), l) for s, e, l in repeating_busy_slots]
        
        merged_busy_slots_for_plot = merge_busy_slots(all_busy_slots_for_day_raw)

        for start_dt, end_dt, label in merged_busy_slots_for_plot:
            start_dt_normalized = datetime.combine(start_date, start_dt.time())
            duration = end_dt - start_dt
            ax.bar(i, duration, bottom=start_dt_normalized, width=0.9, color='grey', alpha=0.3)
            text_y_pos_normalized = start_dt_normalized + duration / 2
            ax.text(i, text_y_pos_normalized, label, ha='center', va='center', color='black', fontsize=8, fontweight='bold', rotation=0, alpha=0.6)
        
        for start, end, name, urgency, source in sorted(schedule.get(date, [])):
            start_dt_normalized = datetime.combine(start_date, start.time())
            duration = end - start
            color = manual_color if source == 'manual' else auto_color
            ax.bar(i, duration, bottom=start_dt_normalized, width=0.6, color=color, edgecolor='black')
            
            text_color = 'black'
            text_y_pos_normalized = start_dt_normalized + duration / 2
            ax.text(i, text_y_pos_normalized, name, ha='center', va='center', color=text_color, fontsize=8, fontweight='bold', rotation=0)

    ax.set_title(f"Your Visual 7-Day Schedule", fontsize=16, fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    fig.tight_layout()
    plt.show()

# --- Event Binding ---
add_busy_slot_button.on_click(add_busy_slot)
add_repeating_slot_button.on_click(add_repeating_busy_slot)
generate_schedule_button.on_click(generate_schedule)
manual_schedule_button.on_click(manual_schedule_task)
clear_schedule_button.on_click(clear_schedule_data)
remove_slot_button.on_click(remove_fixed_slot)

# --- Initial Display ---
display_fixed_times()
refresh_manual_task_dropdown()

# --- GUI Layout ---
add_busy_box = HBox([busy_slot_date_picker, Label("from"), busy_start_input, Label("to"), busy_end_input, busy_label_input, add_busy_slot_button])
add_repeating_box = HBox([Label("Daily from"), repeating_start_input, Label("to"), repeating_end_input, repeating_label_input, add_repeating_slot_button])
remove_slot_box = HBox([remove_slot_dropdown, remove_slot_button])

fixed_time_tabs = Tab(children=[add_repeating_box, add_busy_box, remove_slot_box])
fixed_time_tabs.set_title(0, "Add Daily Slot")
fixed_time_tabs.set_title(1, "Add Specific Slot")
fixed_time_tabs.set_title(2, "Remove a Slot")

manual_schedule_box = HBox([manual_task_dropdown, manual_date_picker, manual_start_time_input, manual_schedule_button])
generate_controls = HBox([generate_schedule_button, clear_schedule_button])

display(VBox([
    Label("Step 1: Manage your fixed appointments.", style={'font_weight': 'bold'}),
    fixed_time_tabs,
    busy_slots_out,
    Label(""),
    Label("Step 2 (Optional): Manually schedule a task.", style={'font_weight': 'bold'}),
    manual_schedule_box,
    Label(""),
    Label("Step 3: Generate or clear your 7-day schedule!", style={'font_weight': 'bold'}),
    generate_controls,
    schedule_out
]))

